# Week 35, Day 4: Comparing encoding models: bits per spike, time rescaling and coupling

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w035/d4.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: LNP against GLM on held-out data

The cell below re-creates yesterday's neuron and fits. Write `loglik(X, w, y)`, the Poisson log-likelihood (dropping the log y! term) for design X with a constant column appended, and compute bits per spike on the held-out last 15 s for both models, relative to a Poisson model with the training mean rate.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
r = np.random.default_rng(2005)
T, D, H = 60000, 20, 40
x = r.normal(size=T); lag = np.arange(D); hl = np.arange(1, H + 1)
k = np.exp(-lag / 4) * np.sin(lag / 2.5); k = 0.6 * k / np.linalg.norm(k)
h = -5 * np.exp(-hl / 2) + 0.8 * np.exp(-((hl - 10) / 4) ** 2)
Xs = np.zeros((T, D))
for j in range(D):
    Xs[j:, j] = x[: T - j]
drive = Xs @ k + np.log(0.03); y = np.zeros(T, int)
for t in range(T):
    y[t] = r.poisson(np.exp(drive[t]))
    if y[t]:
        e = min(T, t + 1 + H); drive[t + 1:e] += y[t] * h[: e - t - 1]
tt = np.log(hl + 3); cs = np.linspace(np.log(3), np.log(H + 3), 6); wd = cs[1] - cs[0]
Bh = np.array([np.where(np.abs(tt - c) < 2 * wd, 0.5 * (1 + np.cos(np.pi * (tt - c) / (2 * wd))), 0) for c in cs]).T
Yh = np.zeros((T, H))
for i in range(1, H + 1):
    Yh[i:, i - 1] = y[: T - i]
Xh = Yh @ Bh

def fit_glm(X, y, iters=25):
    X = np.c_[X, np.ones(len(y))]; w = np.zeros(X.shape[1]); w[-1] = np.log(y.mean())
    for _ in range(iters):
        lam = np.exp(X @ w)
        w = w + np.linalg.solve((X * lam[:, None]).T @ X + 1e-6 * np.eye(len(w)), X.T @ (y - lam))
    return w

train, test = np.arange(T) < 45000, np.arange(T) >= 45000
designs = {"LNP": Xs, "GLM": np.c_[Xs, Xh]}
weights = {m: fit_glm(X[train], y[train]) for m, X in designs.items()}

def loglik(X, w, y):
    eta = np.c_[X, np.ones(len(y))] @ w
    return ...

r0 = y[train].mean()
ll0 = np.sum(y[test] * np.log(r0) - r0)
bps = {m: (loglik(designs[m][test], weights[m], y[test]) - ll0) / (y[test].sum() * np.log(2)) for m in designs}
print({m: round(float(v), 3) for m, v in bps.items()}, "bits per spike on held-out data")

In [ ]:
import numpy as np
def test_positive():
    "both models beat the homogeneous Poisson baseline on held-out data"
    assert bps["LNP"] > 0 and bps["GLM"] > 0

def test_gain():
    "the history terms add more than 0.1 bits per spike"
    assert bps["GLM"] - bps["LNP"] > 0.1

test_positive()
test_gain()
print("All checks passed")

### Exercise: Which model passes?

For each model, compute the predicted intensity on the test set, the corrected rescaled intervals (the intensities summed over the bins strictly between successive test spikes, plus the random share of the later spike's bin), and the KS distance of 1 - exp(-τ) from the uniform distribution.

In [ ]:
def ks_distance(tau):
    z = np.sort(1 - np.exp(-tau)); n = len(z)
    return max((np.arange(1, n + 1) / n - z).max(), (z - np.arange(n) / n).max())

spk = np.where(y[test] > 0)[0]
u = np.random.default_rng(2010).uniform(size=len(spk) - 1)
ks_stat = {}
for m in designs:
    lam = np.exp(np.c_[designs[m][test], np.ones(test.sum())] @ weights[m])
    cum = np.cumsum(lam); ls = lam[spk[1:]]
    between = np.diff(cum[spk]) - ls
    tau = ...
    ks_stat[m] = ks_distance(tau)
crit = 1.63 / np.sqrt(len(spk) - 1)
print({m: round(float(v), 3) for m, v in ks_stat.items()}, f"99% critical value {crit:.3f}")

In [ ]:
import numpy as np
def test_ks():
    "at the 99% level, the LNP fails the time-rescaling test and the GLM passes it"
    assert ks_stat["LNP"] > crit and ks_stat["GLM"] < crit

test_ks()
print("All checks passed")

### Exercise: Recovering a connection

Neuron A is the neuron above. Neuron B has its own stimulus filter and history, plus a coupling filter that adds 1.5 exp(-((t - 3)/1.5)²) to its drive t ms after each spike of A. Fit B with stimulus and history only, and with an added coupling term (A's past spikes projected onto the same basis Bh); compare held-out bits per spike and reconstruct the coupling filter `c_fit`.

In [ ]:
c_true = 1.5 * np.exp(-((hl - 3) / 1.5) ** 2)
kB = np.roll(k, 2)
driveB = Xs @ kB + np.log(0.02); yB = np.zeros(T, int)
for t in range(T):
    yB[t] = r.poisson(np.exp(driveB[t]))
    e = min(T, t + 1 + H)
    if yB[t]:
        driveB[t + 1:e] += yB[t] * h[: e - t - 1]
    if y[t]:
        driveB[t + 1:e] += y[t] * c_true[: e - t - 1]
YhB = np.zeros((T, H))
for i in range(1, H + 1):
    YhB[i:, i - 1] = yB[: T - i]
designs_B = {"uncoupled": np.c_[Xs, YhB @ Bh], "coupled": ...}
wB = {m: fit_glm(X[train], yB[train]) for m, X in designs_B.items()}
r0B = yB[train].mean(); ll0B = np.sum(yB[test] * np.log(r0B) - r0B)
bps_B = {m: (loglik(designs_B[m][test], wB[m], yB[test]) - ll0B) / (yB[test].sum() * np.log(2)) for m in designs_B}
c_fit = Bh @ wB["coupled"][D + 6:D + 12]
print({m: round(float(v), 3) for m, v in bps_B.items()}, "bits per spike for neuron B")
print("coupling, first 8 ms: fitted", np.round(c_fit[:8], 2), " true", np.round(c_true[:8], 2))

In [ ]:
import numpy as np
def test_sign():
    "the fitted coupling from A to B is positive at lags of 2 to 6 ms"
    assert c_fit[1:6].mean() > 0.3

def test_bits():
    "adding coupling improves B's held-out bits per spike"
    assert bps_B["coupled"] > bps_B["uncoupled"] + 0.02

test_sign()
test_bits()
print("All checks passed")